# Extraction de taux de change via l'API OANDA Exchange Rates

Ce notebook se connecte à l'API (`https://api.corporatefxservices.com`), puis collecte des taux spot, un historique journalier, des taux forward, et exporte les résultats.

Source : spécification OpenAPI officielle, version 2.0.0 (copie dans `docs/reference/`).

Avant de commencer :
- Une clé API est nécessaire (essai gratuit de 7 jours sur le site OANDA FX Data Services).
- Elle est envoyée dans l'en-tête `Authorization: Bearer <clé>`.
- Chaque devise de cotation renvoyée compte comme un quote sur votre abonnement.
- Les taux forward, l'open/close et certaines données dépendent de votre abonnement.

In [ ]:
# Dépendances (à lancer une fois si besoin) : requests, pandas, matplotlib, openpyxl
# %pip install requests pandas matplotlib openpyxl

## 1. Connexion

In [ ]:
import os, time, getpass
import requests
import pandas as pd

BASE_URL = "https://api.corporatefxservices.com"

# La clé est lue dans la variable d'environnement OANDA_FX_API_KEY, sinon demandée à la saisie.
# Ne l'écrivez jamais en dur dans le notebook.
API_KEY = os.environ.get("OANDA_FX_API_KEY") or getpass.getpass("Clé API OANDA : ")

session = requests.Session()
session.headers["Authorization"] = f"Bearer {API_KEY}"


def api_get(path, retries=3, **params):
    """Appelle un endpoint (sans extension) et renvoie le JSON décodé.

    Réessaie avec attente croissante sur erreur réseau, 429 et 5xx.
    Lève une erreur explicite sur les autres codes (400, 401, 403, 404...).
    """
    params = {k: v for k, v in params.items() if v is not None}
    for attempt in range(retries):
        try:
            r = session.get(f"{BASE_URL}{path}.json", params=params, timeout=30)
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            if attempt == retries - 1:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue
        if not r.ok:
            raise RuntimeError(f"HTTP {r.status_code} sur {path} : {r.text[:300]}")
        restants = r.headers.get("X-Rate-Limit-Remaining")
        if restants is not None:
            print("Quotes restants :", restants)
        return r.json()

## 2. Test de la connexion

In [ ]:
print(api_get("/v2/remaining_quotes"))

In [ ]:
devises = pd.DataFrame(api_get("/v2/currencies")["currencies"])
print(len(devises), "devises")
devises.head()

## 3. Paramètres : choisissez vos devises et vos dates

Modifiez uniquement cette cellule, puis relancez les suivantes. `BASES` x `QUOTES` donne les paires demandées (chaque devise de cotation compte comme un quote).

In [ ]:
BASES = ["EUR"]                    # devises de base, ex. ["EUR", "USD"]
QUOTES = ["USD", "GBP", "CHF"]     # devises de cotation
DEBUT = "2025-06-01"               # historique : date de début (AAAA-MM-JJ)
FIN = "2026-05-12"                 # historique : date de fin
# Attention : la plage autorisée dépend de votre abonnement. Avec la clé d'essai testée,
# l'API n'acceptait que 2025-05-13 à 2026-05-13 (message d'erreur 400 sinon).
SPOT_DATE_TIME = None              # None = temps réel (non inclus dans l'essai testé, erreur 403)
                                   # sinon un instant passé sur pas de 15 min, ex. "2026-05-12T10:15:00+00:00"
DATA_SET = None                    # None = jeu OANDA par défaut (codes valides via /v2/datasets)
CHAMPS = ("all",)                  # "all" = open, close, moyenne, haut, bas (open/close selon abonnement)

pd.set_option("display.max_rows", None)      # afficher toutes les lignes
pd.set_option("display.max_columns", None)   # afficher toutes les colonnes
pd.set_option("display.width", 200)

## 4. Taux spot (Bid, Ask, Mid)

`base` et `quote` acceptent plusieurs devises. Sans `date_time`, le taux temps réel est renvoyé (mise à jour toutes les 5 secondes). Avec `date_time`, il faut un instant passé sur un pas de 15 minutes, par exemple `2026-05-12T10:15:00+00:00` (accès et plage selon abonnement).

In [ ]:
def spot(base, quote=None, date_time=None, data_set=None):
    try:
        data = api_get("/v2/rates/spot", base=base, quote=quote, date_time=date_time, data_set=data_set)
    except RuntimeError as e:
        if date_time is None and "403" in str(e):
            print("Temps réel indisponible avec cette clé. Renseignez SPOT_DATE_TIME (instant passé) dans les paramètres.")
            return pd.DataFrame()
        raise
    df = pd.DataFrame(data["quotes"])
    for c in ("bid", "ask", "midpoint"):
        df[c] = pd.to_numeric(df[c])
    df["date_time"] = pd.to_datetime(df["date_time"], utc=True)
    return df


taux_spot = spot(base=BASES, quote=QUOTES, date_time=SPOT_DATE_TIME, data_set=DATA_SET)
taux_spot

## 5. Historique journalier

L'endpoint `/v2/rates/candles` renvoie un chandelier par jour (open, close, average, high, low, en bid, ask et midpoint selon `fields`). Une requête est limitée à 10 000 (jours x paires), donc la plage est découpée automatiquement. L'open et le close peuvent exiger un abonnement supérieur.

In [ ]:
def historique(base, quote, debut, fin, fields=("all",), data_set=None):
    base = [base] if isinstance(base, str) else list(base)
    quote = [quote] if isinstance(quote, str) else list(quote)
    paires = len(base) * len(quote)
    jours_max = max(1, min(10_000 // paires, 365))   # limite documentée : 10 000 (jours x paires)

    morceaux = []
    cur = pd.Timestamp(debut)
    fin = pd.Timestamp(fin)
    while cur <= fin:
        stop = min(cur + pd.Timedelta(days=jours_max - 1), fin)
        args = dict(base=base, quote=quote, start_time=cur.strftime("%Y-%m-%dT00:00:00+00:00"),
                    end_time=stop.strftime("%Y-%m-%dT00:00:00+00:00"), data_set=data_set)
        try:
            data = api_get("/v2/rates/candles", fields=list(fields), **args)
        except RuntimeError as e:
            if "403" not in str(e) or "all" not in fields:
                raise
            # L'abonnement n'inclut sans doute pas open/close : repli sur moyenne, haut, bas.
            print("Open/close non disponibles, repli sur averages, highs, lows.")
            data = api_get("/v2/rates/candles", fields=["averages", "highs", "lows"], **args)
        morceaux.append(pd.DataFrame(data["quotes"]))
        cur = stop + pd.Timedelta(days=1)

    df = pd.concat(morceaux, ignore_index=True)
    # Les bornes peuvent se chevaucher d'un morceau à l'autre : on déduplique.
    df = df.drop_duplicates(subset=["base_currency", "quote_currency", "start_time"])
    for c in df.columns:
        if c.endswith(("_bid", "_ask", "_midpoint")):
            df[c] = pd.to_numeric(df[c])
    for c in ("start_time", "open_time", "close_time", "source_date"):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], utc=True)
    return df.sort_values(["base_currency", "quote_currency", "start_time"]).reset_index(drop=True)


hist = historique(BASES, QUOTES, DEBUT, FIN, fields=CHAMPS, data_set=DATA_SET)
print(hist.shape, "lignes x colonnes")
hist   # tableau complet, toutes les colonnes

### Synthèse par paire

In [ ]:
cols = [c for c in ("average_bid", "average_ask", "average_midpoint", "high_midpoint", "low_midpoint") if c in hist.columns]
hist.groupby(["base_currency", "quote_currency"])[cols].agg(["min", "mean", "max"]).round(5)

### Graphiques : un par paire
Ligne bleue : midpoint moyen. Bande : écart entre le plus bas et le plus haut du jour. Lignes fines : bid et ask moyens.

In [ ]:
import matplotlib.pyplot as plt

paires = hist[["base_currency", "quote_currency"]].drop_duplicates().itertuples(index=False)
for b, q in paires:
    d = hist[(hist.base_currency == b) & (hist.quote_currency == q)]
    fig, ax = plt.subplots(figsize=(10, 4))
    if {"low_midpoint", "high_midpoint"} <= set(d.columns):
        ax.fill_between(d["start_time"], d["low_midpoint"], d["high_midpoint"], color="#0000FF", alpha=0.12, label="bas / haut")
    if "average_bid" in d.columns:
        ax.plot(d["start_time"], d["average_bid"], color="#555555", linewidth=0.6, label="bid")
        ax.plot(d["start_time"], d["average_ask"], color="#999999", linewidth=0.6, label="ask")
    ax.plot(d["start_time"], d["average_midpoint"], color="#0000FF", linewidth=1.5, label="midpoint")
    ax.set_title(f"{b}/{q}")
    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Taux")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

## 6. Taux forward (selon abonnement)

`/v2/supported_forwards` liste les paires et échéances disponibles, `/v2/rates/forward` renvoie la courbe pour une paire. Une erreur 403 signifie en général que votre plan n'inclut pas les forwards.

In [ ]:
try:
    forwards_dispos = pd.DataFrame(api_get("/v2/supported_forwards", base=BASES[:1], quote=QUOTES[:1])["supported_forwards"])
    print(forwards_dispos)
    fwd = pd.DataFrame(api_get("/v2/rates/forward", base=BASES[0], quote=QUOTES[0])["quotes"])
    fwd
except RuntimeError as e:
    print("Forwards indisponibles :", e)

## 7. Exportation

In [ ]:
os.makedirs("exports", exist_ok=True)

if not taux_spot.empty:
    taux_spot.to_csv("exports/spot.csv", index=False)
hist.to_csv("exports/historique.csv", index=False)
hist.to_json("exports/historique.json", orient="records", date_format="iso", indent=2)

# Excel : les dates avec fuseau horaire doivent être retirées (UTC conservé dans le nom de colonne)
hist_xl = hist.copy()
for c in hist_xl.select_dtypes(include=["datetimetz"]).columns:
    hist_xl[c] = hist_xl[c].dt.tz_convert("UTC").dt.tz_localize(None)
    hist_xl = hist_xl.rename(columns={c: f"{c}_utc"})
hist_xl.to_excel("exports/historique.xlsx", index=False)

print(os.listdir("exports"))